In [1]:
import json
import os
import re

from num2words import num2words

In [2]:
FILLERS = [
    "jadi gini",
    "jadi begini",
    "gimana ya",
    "apa ya",
    "anu",
    "em",
    "eh",
    "hmm",
]

_FILLER_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(f) for f in FILLERS) + r")\b[,.]?\s*",
    flags=re.IGNORECASE,
)
_PUNCT_PATTERN = re.compile(r"[^\w\s]")
_WHITESPACE_PATTERN = re.compile(r"\s+")
_NUMBER_PATTERN = re.compile(r"\d+")


def remove_punctuation(text: str) -> str:
    return _PUNCT_PATTERN.sub(" ", text)


def remove_fillers(text: str) -> str:
    return _FILLER_PATTERN.sub("", text)


def normalize_numbers(text: str, lang: str = "id") -> str:
    return _NUMBER_PATTERN.sub(lambda m: num2words(int(m.group()), lang=lang), text)


def normalize(text: str) -> str:
    text = remove_punctuation(text)
    text = remove_fillers(text)
    text = normalize_numbers(text)
    text = _WHITESPACE_PATTERN.sub(" ", text).strip().lower()
    return text

In [3]:
DATA_PATH = os.path.join("data", "synthetic_id_formal_informal.jsonl")

records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"loaded {len(records)} records from {DATA_PATH}")

loaded 510 records from data/synthetic_id_formal_informal.jsonl


In [4]:
for r in records:
    r["text_normalized"] = normalize(r["text"])

changed = [r for r in records if r["text"].strip().lower() != r["text_normalized"]]
print(f"{len(changed)} of {len(records)} rows changed by normalization\n")

for r in changed[:10]:
    print("RAW: ", r["text"])
    print("NORM:", r["text_normalized"])
    print()

490 of 510 rows changed by normalization

RAW:  Selamat siang, perkenalkan saya ingin memesan nasi goreng spesial sebanyak 2 porsi untuk meja kami berdua.
NORM: selamat siang perkenalkan saya ingin memesan nasi goreng spesial sebanyak dua porsi untuk meja kami berdua

RAW:  bang gue pesen nasgor spesial dua aja, buat gue sama temen gue nih
NORM: bang gue pesen nasgor spesial dua aja buat gue sama temen gue nih

RAW:  Saya ingin memesan, namun mohon waktu sebentar untuk melihat menu terlebih dahulu karena terdapat banyak pilihan.
NORM: saya ingin memesan namun mohon waktu sebentar untuk melihat menu terlebih dahulu karena terdapat banyak pilihan

RAW:  bentar bentar dulu deh bang, gue liat menunya dulu, banyak banget pilihannya bingung gue
NORM: bentar bentar dulu deh bang gue liat menunya dulu banyak banget pilihannya bingung gue

RAW:  Saya akan memesan ayam bakar madu sebanyak dua porsi beserta nasi putih sebagai pelengkapnya.
NORM: saya akan memesan ayam bakar madu sebanyak dua pors

In [5]:
OUTPUT_PATH = os.path.join("data", "synthetic_id_formal_informal_normalized.jsonl")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"wrote {len(records)} normalized records to {OUTPUT_PATH}")

wrote 510 normalized records to data/synthetic_id_formal_informal_normalized.jsonl
